# 🧪 Sesión 17 · Control de Acceso Basado en Roles (RBAC)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Qué diferencia hay entre no estar autenticado y no tener permiso?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/admin/panel   Authorization: Bearer <token_admin>
HTTP/1.1 200 OK
{"panel":"admin"}

GET /api/admin/panel   Authorization: Bearer <token_user>
HTTP/1.1 403 Forbidden

GET /api/admin/panel   (sin token)
HTTP/1.1 401 Unauthorized

GET /api/perfil        Authorization: Bearer <token_user>
HTTP/1.1 200 OK
{"usuario":"ana","rol":"user"}
```

_El 403 no es un error de credenciales: es una decisión de autorización._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Incluir el rol dentro de los claims del token.
- Distinguir 401 (no autenticado) de 403 (sin permiso).
- Proteger una ruta administrativa con RBAC.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] El administrador obtiene su token → HTTP 200
- [ ] El usuario normal obtiene su token → HTTP 200
- [ ] El admin entra al panel → HTTP 200
- [ ] El usuario normal recibe 403 → HTTP 403
- [ ] Sin token recibe 401 → HTTP 401
- [ ] El perfil funciona para el usuario normal → HTTP 200

## 🧩 El reto

Implementa autenticación con roles: un usuario normal y un administrador. El panel administrativo solo debe abrirse para admin.

**Requisitos del contrato**

- Usuarios de prueba: admin/admin123 (rol admin) y ana/secreta (rol user).
- El token incluye el rol del usuario en sus claims.
- GET /api/admin/panel responde 200 solo para rol admin.
- Un usuario autenticado con rol user recibe 403.
- Sin token se responde 401, no 403.
- GET /api/perfil funciona para cualquier usuario autenticado.

> **Pista conceptual:** Primero autentica (¿quién eres?) y después autoriza (¿tu rol alcanza?). Si devuelves 403 a quien no tiene token, estás mezclando las dos preguntas.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion17.md` con la matriz rol → permiso.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:
%%writefile app.py
from fastapi import FastAPI

app = FastAPI(title="Mi primera implementación RBAC")

# Usuarios de prueba
usuarios = {
    "admin": {"password": "admin123", "rol": "admin"},
    "ana": {"password": "secreta", "rol": "user"}
}

# Iniciar sesión
@app.post("/token")
def iniciar_sesion(username: str, password: str):
    if username in usuarios and usuarios[username]["password"] == password:
        return {
            "access_token": username,
            "token_type": "bearer"
        }

    return {"error": "Usuario o contraseña incorrectos"}

# Consultar perfil
@app.get("/api/perfil")
def consultar_perfil():
    return {"usuario": "ana", "rol": "user"}

# Panel administrativo
@app.get("/api/admin/panel")
def panel_admin():
    return {"panel": "admin"}



Writing app.py


In [10]:


%%writefile app.py

from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import OAuth2PasswordBearer
from jose import jwt, JWTError
from datetime import datetime, timedelta, timezone
from pydantic import BaseModel
import os
import secrets

# Crear aplicación FastAPI
app = FastAPI(title="Sistema de Control de Acceso RBAC")

# Configuración del token JWT
SECRET_KEY = os.getenv("JWT_SECRET_KEY", secrets.token_urlsafe(32))
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

# Usuarios de prueba y sus roles
usuarios = {
    "admin": {"password": "admin123", "rol": "admin"},
    "ana": {"password": "secreta", "rol": "user"}
}


# Modelo para recibir los datos de inicio de sesión
class DatosLogin(BaseModel):
    usuario: str
    password: str


# Configuración de autenticación OAuth2
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/api/token")


# Crear un token con el usuario y su rol
def crear_token(username: str, rol: str):

    expiracion = datetime.now(timezone.utc) + timedelta(
        minutes=ACCESS_TOKEN_EXPIRE_MINUTES
    )

    datos = {
        "usuario": username,
        "rol": rol,
        "exp": expiracion
    }

    return jwt.encode(datos, SECRET_KEY, algorithm=ALGORITHM)


# Obtener token al iniciar sesión
@app.post("/api/token")
def iniciar_sesion(datos: DatosLogin):

    usuario = usuarios.get(datos.usuario)

    # Comprobar usuario y contraseña
    if not usuario or usuario["password"] != datos.password:
        raise HTTPException(
            status_code=401,
            detail="Usuario o contraseña incorrectos",
            headers={"WWW-Authenticate": "Bearer"}
        )

    # Generar token JWT
    access_token = crear_token(
        datos.usuario,
        usuario["rol"]
    )

    return {
        "access_token": access_token,
        "token_type": "bearer"
    }


# Validar token y recuperar los datos del usuario
def obtener_usuario(token: str = Depends(oauth2_scheme)):

    try:
        datos = jwt.decode(
            token,
            SECRET_KEY,
            algorithms=[ALGORITHM]
        )

        username = datos.get("usuario")
        rol = datos.get("rol")

        # Comprobar que el usuario y el rol sean correctos
        if username not in usuarios or rol != usuarios[username]["rol"]:
            raise HTTPException(
                status_code=401,
                detail="Token inválido",
                headers={"WWW-Authenticate": "Bearer"}
            )

        return {
            "usuario": username,
            "rol": rol
        }

    except JWTError:
        raise HTTPException(
            status_code=401,
            detail="Token inválido o vencido",
            headers={"WWW-Authenticate": "Bearer"}
        )


# Consultar perfil: cualquier usuario autenticado puede entrar
@app.get("/api/perfil")
def consultar_perfil(usuario: dict = Depends(obtener_usuario)):

    return {
        "usuario": usuario["usuario"],
        "rol": usuario["rol"]
    }


# Verificar que el usuario tenga rol de administrador
def verificar_admin(usuario: dict = Depends(obtener_usuario)):

    if usuario["rol"] != "admin":
        raise HTTPException(
            status_code=403,
            detail="No tienes permiso para acceder al panel administrativo"
        )

    return usuario


# Panel administrativo: solamente para administradores
@app.get("/api/admin/panel")
def panel_admin(usuario: dict = Depends(verificar_admin)):

    return {
        "panel": "admin"
    }




Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Python 3 y FastAPI en Google Colab, utilizando JWT para la autenticación y control de acceso mediante roles. |
| **Rol** |	Desarrollador especializado en Python, FastAPI y seguridad de aplicaciones web. |
| **Especificación** | Crear dos usuarios: admin y ana, cada uno con su contraseña y rol. Generar tokens JWT para ambos. Permitir que los dos usuarios consulten /api/ perfil, pero restringir /api/ admin/panel únicamente al administrador. Utilizar HTTP 401 cuando no exista un token válido y HTTP 403 cuando el usuario no tenga permisos. |
| **Output** | Archivo app.py completo, comentado y funcional, con las rutas de autenticación, perfil y panel administrativo, cumpliendo los seis criterios de aceptación del verificador. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [11]:
from verificar import probar_http

casos = [{'descripcion': 'El administrador obtiene su token',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'admin', 'password': 'admin123'},
  'estado': 200,
  'guardar': {'token_admin': 'access_token'}},
 {'descripcion': 'El usuario normal obtiene su token',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'guardar': {'token_user': 'access_token'}},
 {'descripcion': 'El admin entra al panel',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_admin}'},
  'estado': 200,
  'json_contiene': {'panel': 'admin'}},
 {'descripcion': 'El usuario normal recibe 403',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 403},
 {'descripcion': 'Sin token recibe 401', 'metodo': 'GET', 'ruta': '/api/admin/panel', 'estado': 401},
 {'descripcion': 'El perfil funciona para el usuario normal',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana', 'rol': 'user'}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 6 casos · app.py
✅ 01. El administrador obtiene su token
✅ 02. El usuario normal obtiene su token
✅ 03. El admin entra al panel
✅ 04. El usuario normal recibe 403
✅ 05. Sin token recibe 401
✅ 06. El perfil funciona para el usuario normal
------------------------------------------------------------------------------
RESULTADO: 6/6 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion17.md` a `auditorias/auditoria_sesion17.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?

   Lo que más me costó fue conseguir que los usuarios pudieran obtener su token correctamente, porque al principio las rutas y los nombres de los campos no coincidían con los que esperaba el verificador. Tuve que corregir la ruta de inicio de sesión y adaptar los datos que recibía la aplicación.

   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?

   Decidí separar la validación del token y la comprobación de permisos del administrador mediante funciones diferentes. De esta manera, cualquier usuario autenticado puede consultar su perfil, pero solamente el administrador puede entrar al panel.

   - ¿Qué error detectó la verificación que tú no habías previsto?

   La verificación detectó que mi aplicación esperaba el campo username, pero el verificador enviaba usuario. Esto provocaba un error HTTP 422. También detectó inicialmente que la ruta de inicio de sesión no coincidía con la solicitada. Con estas pruebas pude identificar y corregir los problemas.
   
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
